In [1]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


## Langchain Integration
https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html#langchain-integration

### Harmonized Model Initialization
The init_llm and init_embedding_model functions allow easy initialization of langchain model interfaces in a harmonized way in generative AI hub sdk

In [2]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from gen_ai_hub.proxy.langchain.init_models import init_llm

template = """Question: {question}
    Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=['question'])
question = 'What is a supernova?'

llm = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({'question': question})
print(response)

A supernova is a powerful and luminous explosion that occurs at the end of a star's life cycle. Let's break it down step by step:

1. **Star's Life Cycle**: Stars are massive celestial bodies composed primarily of hydrogen and helium. They generate energy through nuclear fusion, converting hydrogen into helium in their cores. This process releases a tremendous amount of energy, which counteracts the gravitational forces trying to collapse the star.

2. **End of Fusion**: As a star exhausts its nuclear fuel, it can no longer sustain fusion reactions in its core. For massive stars, this typically means they have fused elements up to iron, beyond which fusion is not energetically favorable.

3. **Core Collapse**: Without the outward pressure from fusion, the core of the star begins to collapse under its own gravity. This collapse happens extremely rapidly, often in a matter of seconds.

4. **Explosion**: The core collapse results in a rebound effect, where the outer layers of the star are

init_embedding_model

In [3]:
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

text = 'Every decoding is another encoding.'

embeddings = init_embedding_model('text-embedding-3-large')
response = embeddings.embed_query(text)
#print(response)


### Chat model

In [4]:
from langchain_core.prompts.chat import (
    AIMessagePromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

chat_llm = ChatOpenAI(proxy_model_name='gpt-4o', proxy_client=proxy_client)

template = 'You are a helpful assistant that translates english to Chinese.'
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

example_human = HumanMessagePromptTemplate.from_template('Hi')
example_ai = AIMessagePromptTemplate.from_template('Ahoy!')
human_template = '{text}'

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)
chat_prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, example_human, example_ai, human_message_prompt])

chain = chat_prompt | chat_llm

response = chain.invoke({'text': 'I love planking.'})
print(response.content)


我喜欢平板支撑。


### Structured model outputs

In [5]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.prompts.chat import HumanMessage
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
chat_model = ChatOpenAI(proxy_model_name="gpt-4o", proxy_client=get_proxy_client())
chat_model = chat_model.with_structured_output(method="json_schema", schema=Person, strict=True)

message = HumanMessage(content="Tell me about a person named John who is 30")
print(chat_model.invoke([message]))


name='John Doe' age=30


## Agent

### Basic structure

#### Define tools

In [21]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Define agents

In [22]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [93]:

response =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "What is weather in Shanghai?"
            }
        ]
    }
)
print(response)

 

{'messages': [HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='08208c2f-5637-4770-b75a-56075bc7d3b4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_enZUAh8J7rtbu8Gmz10z1GKv', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-Cney9OvMlY5Za777mVTiZge16h3wu', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2aea-d8f2-7880-aa7c-ee6fa975c560-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'c

Define a function to return message directly.

In [94]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  # 若缺失会直接抛 KeyError


To check the contect in an easier way, we create a function to find out and then print out key information based on the structure of this message.

In [48]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        "assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




Now it is easier for us to see that, in the below case the tool [search] is not applied.

In [102]:
messages = invoke_agent_messages(agent, "Where is the location of Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Where is the location of Shanghai?",
  "tool_name": null,
  "tool_output": null,
  "assistant_text": "Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea."
}


To let agent to use the tool, change the question closer to the tool description.

In [103]:
messages = invoke_agent_messages(agent, "Search for the location of Shanghai")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Search for the location of Shanghai",
  "tool_name": "search",
  "tool_output": "Results for: location of Shanghai",
  "assistant_text": "Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea."
}


### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [11]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [18]:
query="Search for the explaination of transformers in NLP"

In [19]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

Transformers in Natural Language Processing (NLP) are a type of neural network architecture designed to handle sequential data, such as text, more efficiently than previous models like RNNs (Recurrent Neural Networks). Here's a simple explanation of how they work and why they are important:

1. **Self-Attention Mechanism**: Transformers use a mechanism called self-attention, which allows the model to weigh the importance of different words in a sentence when making predictions. This means that the model can focus on relevant parts of the input sequence, regardless of their position, which is crucial for understanding context.

2. **Parallelization**: Unlike RNNs, which process data sequentially, transformers can process entire sequences of data at once. This parallelization makes them much faster and more efficient, especially for long sequences.

3. **Encoder-Decoder Structure**: Transformers are often structured with an encoder and a decoder. The encoder processes the input data and 

In [20]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

Transformers are a type of neural network architecture that has revolutionized the field of Natural Language Processing (NLP). They were introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017. Here are the key components and concepts of transformers in NLP:

1. **Self-Attention Mechanism**: 
   - The core innovation of transformers is the self-attention mechanism, which allows the model to weigh the importance of different words in a sentence when encoding a particular word. This mechanism helps the model understand the context and relationships between words, regardless of their position in the sentence.

2. **Multi-Head Attention**:
   - Transformers use multiple attention heads to capture different types of relationships and features from the input data. Each head processes the input independently, and their outputs are combined to provide a richer representation.

3. **Positional Encoding**:
   - Since transformers do not inherently understand the order of wo

### Decorater

#### Setup: model + tools

In [89]:
from langchain_core.tools import tool
# --- Define tools ---
@tool
# def search(query: str) -> str:
#     """Search for information."""
#     return f"Results for: {query}"
# @tool
def tool1(query: str) -> str:
    """A demo tool that echoes the query with a tag."""
    return f"[query] You asked: {query}"



@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

# --- Define model (replace your API key/config as needed) ---
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=4800
)

#### Define custom state + middleware

In [81]:
#from langchain_openai import ChatOpenAI
from typing import Any, TypedDict
from langchain.agents.middleware import AgentMiddleware


class CustomState(AgentState):
    # You can add arbitrary fields to state beyond messages:
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    # Declare which state schema this middleware expects:
    state_schema = CustomState

    # Optionally restrict/override tools visible at this middleware stage:
    tools = [search, get_weather]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        """
        Runs before every model call.
        Read user_preferences (if any) and inject a system prompt + model kwargs.
        Return a dict to merge into the model request (e.g. messages, model_kwargs).
        """
        # Safely read user_preferences from state (dict-like or attribute)
        prefs = (state.get("user_preferences") or {}) if isinstance(state, dict) \
            else getattr(state, "user_preferences", {}) or {}

        style = str(prefs.get("style", "general")).lower()         # e.g., "technical" or "casual"
        verbosity = str(prefs.get("verbosity", "normal")).lower()  # e.g., "detailed" or "brief"

        # Build a dynamic system prompt based on preferences
        system_prompt = "You are a helpful assistant."
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal and approachable."

        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points."

        # Optionally tune generation parameters
        temperature = 0.2 if style == "technical" else 0.7

        # Return updates to be merged into the model request
        return {
            # Safest cross-version approach: inject a SYSTEM message
            "messages": [{"role": "system", "content": system_prompt}],
            # Adjust model kwargs (field name may vary by version; this is common)
            "model_kwargs": {"temperature": temperature},
        }


#### Create the agent

In [90]:
from langchain.agents import create_agent, AgentState

tools = [search, get_weather]

agent = create_agent(
    model,
    tools=tools,
    middleware=[CustomMiddleware()],  # Plug in your middleware
    system_prompt="You are a helpful assistant. Be concise and accurate."
)


#### Invoke the agent with user_preferences

In [111]:
query="Search for the explaination vector embeddings." 
query="Search for the story lines and theme in 三国演义 in Chinese" 
query="Search for the story lines and theme in Games of Throne and then introduce these in Chinese." 

In [112]:
# A user who prefers technical & detailed responses
result = agent.invoke({
    "messages": [{"role": "user", "content": query}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

messages = result["messages"]

print_message_pairs(messages,verbose=True)
print("\n=============================== Assistant reply (technical + detailed) =================================")
print_message_pairs(messages)


{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes",
  "assistant_text": "**Game of Thrones Storylines and Themes:**\n\n**Storylines:**\n1. **The Iron Throne:** The central storyline revolves around the struggle for power and control of the Iron Throne of the Seven Kingdoms. Various noble families, including the Starks, Lannisters, Baratheons, and Targaryens, vie for dominance.\n2. **The Stark Family:** The Stark family of Winterfell faces numerous challenges, including betrayal, loss, and the quest for justice. Key members like Eddard Stark, Jon Snow, and Arya Stark play significant roles.\n3. **Daenerys Targaryen's Quest:** Daenerys Targaryen's journey from exile to becoming a powerful leader with dragons, aiming to reclaim the Iron Throne.\n4. **The Night's Watch and the White Walkers:** The threat of the White Walkers and the N

In [113]:

# A user who prefers casual & brief responses
result = agent.invoke({
    "messages": [{"role": "user", "content": query}],
    "user_preferences": {"style": "casual", "verbosity": "brief"},
})

messages = result["messages"]

print_message_pairs(messages,verbose=True)
print("\n=============================== Assistant reply (casual + brief) =================================")
print_message_pairs(messages)



{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes",
  "assistant_text": "\"Game of Thrones\" is a fantasy drama series based on George R.R. Martin's \"A Song of Ice and Fire\" novels. The storylines revolve around the power struggles among noble families in the fictional continents of Westeros and Essos. Key themes include the quest for power, betrayal, loyalty, and the impact of war on society. The series also explores complex characters, moral ambiguity, and the consequences of political decisions.\n\nNow, let's introduce these in Chinese:\n\n《权力的游戏》是一部根据乔治·R·R·马丁的小说《冰与火之歌》改编的奇幻剧集。故事情节围绕在虚构大陆维斯特洛和厄索斯上贵族家族之间的权力斗争。主要主题包括权力的追求、背叛、忠诚以及战争对社会的影响。该剧还探讨了复杂的人物、道德的模糊性以及政治决策的后果。"
}

=============================== Assistant reply (casual + brief) =================================
"Game of Thrones" is a fantasy drama series based on George 

#### Invoke the agent with user_preferences

#### Advanced concepts

##### ToolStrategy

ToolStrategy uses artificial tool calling to generate structured output. This works with any model that supports tool calling:

In [131]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    tools=[search],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]
# # ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

##### Memory

In [143]:

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any

# 假设已有 model, tool1, tool2, tools

class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = [search, get_weather]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        prefs = (state.get("user_preferences") or {}) if isinstance(state, dict) else getattr(state, "user_preferences", {}) or {}
        style = prefs.get("style", "general")
        verbosity = prefs.get("verbosity", "normal")

        system_prompt = "You are a helpful assistant."
        if style == "technical":
            system_prompt += " Prefer precise, technical language and cite implementation details where relevant."
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with examples."

        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": 0.2 if style == "technical" else 0.7}
        }

agent = create_agent(
    model,
    tools=[search, get_weather],                     # 全局工具
    middleware=[CustomMiddleware()]  # 中间件（限定阶段性工具 + 注入系统提示）
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

# 读取最终回复（具体键按你的返回结构而定）
messages = result["messages"]
print(messages[-1]["content"] if isinstance(messages[-1], dict) else messages[-1].content)



Understood! Whenever you have a question or need an explanation, I'll provide detailed, technical insights, including implementation details and examples where applicable. If you have a specific topic or question in mind, feel free to ask, and I'll dive into the technical aspects for you.


In [148]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Introduce transformers in NLP in Chinese."}],
    "user_preferences": {"style": "technical", "verbosity": "normal"},
})

# 读取最终回复（具体键按你的返回结构而定）
messages = result["messages"]
print(messages[-1]["content"] if isinstance(messages[-1], dict) else messages[-1].content)

Transformer 是一种用于自然语言处理（NLP）的深度学习模型架构，由 Vaswani 等人在 2017 年提出。它通过自注意力机制和并行化处理来提高模型的效率和性能。以下是 Transformer 的一些关键特点和组件：

1. **自注意力机制（Self-Attention Mechanism）**：
   - 自注意力机制允许模型在处理输入序列时关注序列中的不同位置。这种机制使得模型能够捕捉到词与词之间的关系，而不依赖于序列的距离。
   - 通过计算输入序列中每个词的注意力权重，模型可以动态地调整对不同词的关注程度。

2. **多头注意力（Multi-Head Attention）**：
   - 多头注意力机制通过并行化多个自注意力层来捕捉不同的特征表示。每个注意力头可以学习不同的关系和模式，从而增强模型的表达能力。

3. **位置编码（Positional Encoding）**：
   - 由于 Transformer 不使用循环结构（如 RNN），它需要一种机制来表示输入序列中词的位置。位置编码通过向输入向量添加位置信息来解决这个问题。

4. **编码器-解码器结构（Encoder-Decoder Architecture）**：
   - Transformer 包含编码器和解码器两个部分。编码器负责处理输入序列并生成上下文表示，解码器则利用这些表示生成输出序列。
   - 编码器由多个相同的层组成，每层包含多头注意力和前馈神经网络。解码器的结构类似于编码器，但在多头注意力层中加入了对编码器输出的注意力。

5. **并行化处理**：
   - Transformer 的架构允许并行处理输入数据，这显著提高了训练速度和效率，尤其是在处理长序列时。

Transformer 模型在许多 NLP 任务中取得了显著的成功，包括机器翻译、文本生成和问答系统。其代表性实现包括 BERT、GPT 系列和 T5 等模型，这些模型在各种基准测试中表现优异。


In [149]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Introduce transformers in NLP in Chinese."}],
    "user_preferences": {"style": "normal", "verbosity": "detailed"},
})

# 读取最终回复（具体键按你的返回结构而定）
messages = result["messages"]
print(messages[-1]["content"] if isinstance(messages[-1], dict) else messages[-1].content)

在自然语言处理（NLP）领域，"transformers"（变压器）是一种深度学习模型架构，它在处理序列数据方面表现出色，尤其是在语言任务中。变压器模型由Vaswani等人在2017年提出，并迅速成为NLP领域的主流技术。以下是对变压器模型的详细介绍：

### 变压器模型的核心思想

变压器模型的核心思想是通过注意力机制（Attention Mechanism）来处理序列数据。传统的序列模型，如循环神经网络（RNN）和长短时记忆网络（LSTM），通常依赖于序列的顺序处理，而变压器则通过注意力机制允许模型在处理序列时关注序列中的不同部分，而不必严格按照顺序进行。

### 变压器的结构

变压器模型主要由编码器（Encoder）和解码器（Decoder）两部分组成：

1. **编码器（Encoder）**：
   - 编码器由多个相同的层堆叠而成，每一层包括两个主要部分：多头自注意力机制（Multi-Head Self-Attention）和前馈神经网络（Feed-Forward Neural Network）。
   - 自注意力机制允许模型在处理输入序列的每个位置时，关注序列中的其他位置，从而捕捉到序列中的长距离依赖关系。
   - 前馈神经网络则对每个位置的输出进行进一步的处理。

2. **解码器（Decoder）**：
   - 解码器的结构与编码器类似，但在每一层中增加了一个额外的注意力机制，用于关注编码器的输出。
   - 解码器的自注意力机制是掩蔽的（Masked），以确保解码器在生成序列时只能关注已经生成的部分。

### 注意力机制

注意力机制是变压器模型的核心组件，它通过计算输入序列中每个位置与其他位置的相关性来决定应该关注哪些部分。注意力机制的计算包括以下步骤：

1. **计算查询（Query）、键（Key）和值（Value）**：
   - 对输入进行线性变换，得到查询、键和值向量。

2. **计算注意力得分**：
   - 通过查询和键的点积计算注意力得分，并通过Softmax函数进行归一化。

3. **加权求和**：
   - 使用注意力得分对值向量进行加权求和，得到最终的注意力输出。

### 多头注意力机制

多头注意力机制通过并行计算多个注意力机制来捕捉不同的特征。每个注意力头独立地执行上述注意力计算，然后将所有头的输出

In [ ]:

class CustomState(AgentState):
    # You can add arbitrary fields to state beyond messages:
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    # Declare which state schema this middleware expects:
    state_schema = CustomState

    # Optionally restrict/override tools visible at this middleware stage:
    tools = [tool1, tool2]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        """
        Runs before every model call.
        Read user_preferences (if any) and inject a system prompt + model kwargs.
        Return a dict to merge into the model request (e.g. messages, model_kwargs).
        """
        # Safely read user_preferences from state (dict-like or attribute)
        prefs = (state.get("user_preferences") or {}) if isinstance(state, dict) \
            else getattr(state, "user_preferences", {}) or {}

        style = str(prefs.get("style", "general")).lower()         # e.g., "technical" or "casual"
        verbosity = str(prefs.get("verbosity", "normal")).lower()  # e.g., "detailed" or "brief"

        # Build a dynamic system prompt based on preferences
        system_prompt = "You are a helpful assistant."
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal and approachable."

        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points."

        # Optionally tune generation parameters
        temperature = 0.2 if style == "technical" else 0.7

               # Return updates to be merged into the model request
        return {
            # Safest cross-version approach: inject a SYSTEM message
            "messages": [{"role": "system", "content": system_prompt}],
            # Adjust model kwargs (field name may vary by version; this is common)
            "model_kwargs": {"temperature": temperature},
        }


In [ ]:
def demo_args(*args):
    print(args)  # args 是一个元组
    for i, value in enumerate(args, start=1):
        print(f"第{i}个参数: {value}")

import jsondemo_args(10, 20, 30)

(10, 20, 30)
第1个参数: 10
第2个参数: 20
第3个参数: 30


In [31]:
import json

def demo_kwargs(**kwargs):
    print(kwargs)  # kwargs 是一个字典
    for key1, value2 in kwargs.items():
        print(f"{key1} = {value2}")
   
    print(json.dumps(kwargs)) 
demo_kwargs(name="Alice", age=25, city="Shanghai")

{'name': 'Alice', 'age': 25, 'city': 'Shanghai'}
name = Alice
age = 25
city = Shanghai
{"name": "Alice", "age": 25, "city": "Shanghai"}


In [41]:
class MyClass:
    @staticmethod
    def static_method():
        print("This is a static method.")

    @classmethod
    def class_method(cls):
        print(f"This is a class method of {cls.__name__}.")

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

# 使用
MyClass.static_method()
MyClass.class_method()

obj = MyClass()
obj.name = "Alice"
print(obj.name)

This is a static method.
This is a class method of MyClass.
Alice
